# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# 1. Đọc dữ liệu từ data/splits/train.jsonl
train_path = "data/splits/train.jsonl"
with open(train_path, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

total_images = len(train_data)
total_boxes = sum(len(item["objects"]) for item in train_data)
print(f"Tổng số ảnh tập train: {total_images}")
print(f"Tổng số bounding boxes tập train: {total_boxes}")

# 2. Trực quan hóa 6 ảnh mẫu
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx in range(min(6, total_images)):
    item = train_data[idx]
    img_path = os.path.join("data", item["image_path"])
    img = Image.open(img_path)
    w_img, h_img = item["width"], item["height"]

    ax = axes[idx]
    ax.imshow(img)
    ax.set_title(f"ID: {item['image_id'][:10]}...", fontsize=10)
    ax.axis("off")

    # Vẽ từng bbox
    for obj in item["objects"]:
        label = obj["label"]
        xc, yc, w, h = obj["yolo"]
        
        # Quy đổi tọa độ YOLO [0, 1] sang Toạ độ Pixel [x1, y1, width, height]
        x1 = (xc - w / 2) * w_img
        y1 = (yc - h / 2) * h_img
        bw = w * w_img
        bh = h * h_img

        rect = patches.Rectangle((x1, y1), bw, bh, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, max(0, y1 - 5), label, color="white", fontsize=9, 
                weight="bold", bbox=dict(facecolor="red", alpha=0.7, pad=1, edgecolor="none"))

plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

# Định nghĩa Prompt hệ thống dùng xuyên suốt các bài thí nghiệm
PROMPT = (
    "Detect all defects (pothole, crack, manhole) in the image. "
    "Return a JSON list of objects with 0-1000 normalized coordinates in the format: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}].'
)

def yolo_to_box(label: str, yolo: list) -> dict:
    """
    Chuyển nhãn YOLO [xc, yc, w, h] thuộc [0, 1] sang box [x1, y1, x2, y2] thang integer 0–1000.
    """
    xc, yc, w, h = yolo
    
    # Quy đổi tọa độ
    x1 = int(round((xc - w / 2) * 1000))
    y1 = int(round((yc - h / 2) * 1000))
    x2 = int(round((xc + w / 2) * 1000))
    y2 = int(round((yc + h / 2) * 1000))

    # Kẹp vào khoảng [0, 1000]
    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))

    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects: list) -> str:
    """
    Chuyển danh sách objects sang chuỗi JSON rút gọn, sắp xếp thứ tự ưu tiên y1 -> x1.
    """
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    # Sắp xếp top -> bottom, left -> right để model học định dạng cố định
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# --- Test nhanh kiểm tra logic ---
sample_objs = [{"label": "pothole", "yolo": [0.5, 0.5, 0.2, 0.2]}]
print("Test output JSON:", objects_to_answer(sample_objs))

## 3. E0: zero-shot

In [ ]:
import os
import json
import torch
from tqdm import tqdm
from PIL import Image
from transformers import BitsAndBytesConfig, AutoProcessor, Qwen2_5_VLForConditionalGeneration

# 1. Cấu hình lượng tử hóa 4-bit (giữ visual & lm_head ở fp16)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

model_id = "Qwen/Qwen3-VL-4B-Instruct"  # Sử dụng tên mô hình chính xác theo bài lab

print("Đang tải model và processor...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"

# 2. Đọc tập test
test_path = "data/splits/test.jsonl"
with open(test_path, "r", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f]

os.makedirs("work", exist_ok=True)
output_path = "work/e0.jsonl"

# 3. Chạy suy luận Zero-Shot
model.eval()
results = []

print(f"Đang thực hiện Zero-shot E0 trên {len(test_data)} ảnh...")
with torch.no_grad():
    for item in tqdm(test_data):
        img_full_path = os.path.join("data", item["image_path"])
        image = Image.open(img_full_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT}
                ]
            }
        ]

        # Áp dụng template và chuẩn bị inputs
        text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text_prompt], images=[image], return_tensors="pt", padding=True)
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

        prompt_length = inputs["input_ids"].shape[1]

        # Sinh câu trả lời
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        
        # Chỉ lấy phần output được sinh mới
        raw_output = processor.decode(out[0, prompt_length:], skip_special_tokens=True)

        results.append({
            "image_id": item["image_id"],
            "raw": raw_output
        })

# 4. Ghi kết quả vào file work/e0.jsonl
with open(output_path, "w", encoding="utf-8") as f:
    for res in results:
        f.write(json.dumps(res, ensure_ascii=False) + "\n")

print(f"Đã hoàn thành E0. Kết quả ghi tại: {output_path}")

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence

def encode_example(image, prompt: str, answer: str, processor):
    """
    Mã hóa 1 mẫu dữ liệu và gắn mask loss (-100) cho phần prompt và token ảnh.
    """
    user_msg = {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]
    }
    assistant_msg = {
        "role": "assistant",
        "content": answer
    }

    # 1. Encode prompt
    prompt_str = processor.apply_chat_template([user_msg], tokenize=False, add_generation_prompt=True)
    prompt_enc = processor(text=[prompt_str], images=[image], return_tensors="pt")

    # 2. Encode chuỗi đầy đủ (Prompt + Answer)
    full_str = processor.apply_chat_template([user_msg, assistant_msg], tokenize=False, add_generation_prompt=False)
    full_enc = processor(text=[full_str], images=[image], return_tensors="pt")

    # 3. Kiểm tra tiền tố
    n_prompt_tokens = prompt_enc.input_ids.shape[1]
    assert torch.equal(full_enc.input_ids[0, :n_prompt_tokens], prompt_enc.input_ids[0]), "Mismatch prompt prefix!"

    # 4. Mask Loss: Đặt -100 cho prompt và token vision
    labels = full_enc.input_ids.clone()
    labels[0, :n_prompt_tokens] = -100

    example = {k: v[0] for k, v in full_enc.items()}
    example["labels"] = labels[0]
    return example


def collate_fn(batch, processor):
    """
    Gộp các mẫu thành batch, thực hiện padding văn bản và concatenate đặc trưng ảnh.
    """
    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0

    # Padding chuỗi văn bản & labels
    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=pad_id)
    attention_mask_padded = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)

    batch_out = {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "labels": labels_padded
    }

    # Gộp token loại đa phương thức (nếu có)
    if "mm_token_type_ids" in batch[0]:
        mm_ids = [item["mm_token_type_ids"] for item in batch]
        batch_out["mm_token_type_ids"] = pad_sequence(mm_ids, batch_first=True, padding_value=0)

    # Nối đặc trưng hình ảnh theo chiều batch (KHÔNG pad)
    if "pixel_values" in batch[0]:
        batch_out["pixel_values"] = torch.cat([item["pixel_values"] for item in batch], dim=0)
    if "image_grid_thw" in batch[0]:
        batch_out["image_grid_thw"] = torch.cat([item["image_grid_thw"] for item in batch], dim=0)

    return batch_out

print("Đã hoàn tất định nghĩa encode_example() và collate_fn()!")

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
import torch
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup: str):
    """
    Áp dụng cấu hình đóng/mở tham số và LoRA adapter cho các bài thí nghiệm E1, E2, E3.
    """
    # 1. Đóng băng tất cả các tham số của mô hình gốc
    for param in model.parameters():
        param.requires_grad = False
        
    # Bật tính năng tính gradient cho input (bắt buộc để sử dụng gradient checkpointing)
    model.enable_input_require_grads()

    # 2. Xกำหนด các module target cho LoRA
    # Các khối LLM áp dụng cho cả E1, E2, E3 (252 modules)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
    
    if setup == "E3":
        # Thêm các module ViT cho E3 (thêm 96 modules -> Tổng 348 modules)
        target_modules += ["qkv", "proj", "linear_fc1", "linear_fc2"]

    # Cấu hình LoRA adapter
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_config)

    # 3. Mở khóa toàn bộ tham số projector (visual.merger) cho E2 và E3
    if setup in ["E2", "E3"]:
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                # Đưa về float32 để đảm bảo độ chính xác khi huấn luyện đầy đủ
                param.data = param.data.to(torch.float32)
                param.requires_grad = True

    # Thống kê số lượng tham số có thể huấn luyện
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup}] Trainable params: {trainable_params:,} / {all_params:,} ({100 * trainable_params / all_params:.2f}%)")

    return peft_model

print("Đã định nghĩa xong hàm apply_setup()!")

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9